In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import faiss
import wandb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath('..'))
from src.utils import map_at_3, load_config, set_seed

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
TRAIN_PATH  = '../data/raw/train.csv'

train = pd.read_csv(TRAIN_PATH)
print(f'Shape: {train.shape}')

/Users/risshabsrinivas/Documents/IITM/DL_GENAI_PROJECT/dl-genai-project-26-t2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (2000, 8)


In [2]:
# ── Knowledge Base + FAISS Index ──────────────────────────────────────────────
print('Building knowledge base...')
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print('Loading embedding model and creating FAISS index...')
bi_encoder    = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = bi_encoder.encode(kb, show_progress_bar=True)
kb_embeddings = np.array(kb_embeddings).astype('float32')

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f'Knowledge base size : {len(kb)}')
print(f'FAISS index size    : {index.ntotal}')
print('Knowledge base successfully created.')

Building knowledge base...
Loading embedding model and creating FAISS index...


Batches: 100%|██████████| 63/63 [00:04<00:00, 14.65it/s]

Knowledge base size : 2000
FAISS index size    : 2000
Knowledge base successfully created.
